In [1]:
import torch
import torch.nn.functional as F

In [4]:
words = open('../names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [5]:
def build_dataset(words, block_size):
    X, y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            y.append(ix)
            context = context[1:] + [ix] 

    X = torch.tensor(X)
    y = torch.tensor(y)

    return X, y

In [6]:
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
block_size = 3

Xtr, ytr = build_dataset(words[:n1], block_size)
Xdev, ydev = build_dataset(words[n1:n2], block_size)
Xte, yte = build_dataset(words[n2:], block_size)

In [46]:
class MLP:
    def __init__(self, embed_size: int, hidden_layer_size: int, block_size: int,seed: int = -1):
        # Seed
        self.seed = seed if seed != -1 else torch.randint(low=1, high=50000, size=()).item()
        self.g = torch.Generator().manual_seed(self.seed)

        # Object info
        self.embed_size = embed_size
        self.hidden_layer_size = hidden_layer_size
        self.block_size = block_size

        # Initialize parameters
        self.C = torch.randn((27, embed_size), generator=self.g)
        self.W1 = torch.randn((embed_size * block_size, hidden_layer_size), generator=self.g) * ((5/3) / (embed_size * block_size)**0.5) #norm
        self.W2 = torch.randn((hidden_layer_size, 27), generator=self.g) * 0.01
        self.b2 = torch.randn(27, generator=self.g) * 0

        self.bngain = torch.ones((1, hidden_layer_size))
        self.bnbias = torch.zeros((1, hidden_layer_size))

        self.bnmean = torch.zeros((1, hidden_layer_size))
        self.bnstd = torch.ones((1, hidden_layer_size))

        self.parameters = [self.C, self.W1, self.W2, self.b2, self.bngain, self.bnbias]
        for p in self.parameters:
            p.requires_grad = True

    def train(self, X: torch.Tensor, y: torch.Tensor, num_iterations: int = 10000, batch_size: int = 32, learn_rate: list[float] = [0.1, 0.02, 0.005]):
        for lr in learn_rate:
            for _ in range(num_iterations):
                #batch construct
                ix = torch.randint(0, X.shape[0], (batch_size, ))
            
                # forward pass
                logits = self._forward(X=X[ix], training=True, num_iterations=num_iterations)
                loss = F.cross_entropy(logits, y[ix])

                #backward pass
                for p in self.parameters:
                    p.grad = None
                loss.backward()

                #update
                for p in self.parameters:
                    p.data += -lr * p.grad

        print("Train loss: ", self.eval(X, y))

    @torch.no_grad
    def eval(self, X: torch.Tensor, y: torch.Tensor):
        logits = self._forward(X=X)
        loss = F.cross_entropy(logits, y)
        return loss.item()

    @torch.no_grad
    def generate(self, num_words: int):
        ret = []
        for _ in range(num_words):
            out = []
            context = [0] * self.block_size
            while True:
                logits = self._forward(torch.tensor([context]))
                probs = F.softmax(logits, dim=1)
                ix = torch.multinomial(probs, num_samples=1, generator=self.g).item()
                context = context[1:] + [ix]
                out.append(ix)
                if ix == 0:
                    break

            ret.append(''.join(itos[i] for i in out))

        return ret

    def _forward(self, X: torch.Tensor, training: bool = False, num_iterations: int = 10000):
        emb = self.C[X]
        emb_concat = emb.view(-1, self.embed_size * self.block_size)
        h_preact = emb_concat @ self.W1
        if training:
            bn_mean = h_preact.mean(0, keepdim=True)
            bn_std = h_preact.std(0, keepdim=True)
            h_preact = self.bngain * (h_preact - bn_mean) / bn_std + self.bnbias
            with torch.no_grad():
                self.bnmean = 0.999 * self.bnmean + 0.001 * bn_mean
                self.bnstd = 0.999 * self.bnstd + 0.001 * bn_std
        else:
            h_preact = self.bngain * (h_preact - self.bnmean) / self.bnstd + self.bnbias
        h = torch.tanh(h_preact)
        logits = h @ self.W2 + self.b2
        return logits

    def num_params(self):
        return sum(p.nelements() for p in self.parameters)

In [47]:
model = MLP(embed_size=10, hidden_layer_size=200, block_size=block_size, seed=42)
model.train(Xtr, ytr, num_iterations=100000, learn_rate=[0.15, 0.03, 0.005])
print("Dev loss for model:", model.eval(Xdev, ydev))

Train loss:  2.0707085132598877
Dev loss for model: 2.1114046573638916


In [50]:
model.generate(num_words=5)

['cin.', 'plai.', 'eraming.', 'kemeiton.', 'alix.']

In [58]:
# Deeper network
g = torch.Generator().manual_seed(42)

class Linear:

    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)
            xvar = x.var(0, keepdim=True, unbiased=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
            return []

n_embed = 10
n_hidden = 100
vocab_size = 27

C = torch.randn((vocab_size, n_embed), generator=g)
layers = [
    Linear(n_embed * block_size, n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size), BatchNorm1d(vocab_size),
]

with torch.no_grad():
    layers[-1].gamma *= 0.1
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

47351


In [59]:
max_steps = 200000
batch_size = 32

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, yb = Xtr[ix], ytr[ix]

    emb = C[Xb]
    x = emb.view(emb.shape[0], -1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x, yb)

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')

      0/ 200000: 3.2982
  10000/ 200000: 2.1165
  20000/ 200000: 2.0326
  30000/ 200000: 2.1251
  40000/ 200000: 2.1984
  50000/ 200000: 2.0994
  60000/ 200000: 2.1105
  70000/ 200000: 2.1281
  80000/ 200000: 1.6997
  90000/ 200000: 2.3428
 100000/ 200000: 1.8410
 110000/ 200000: 1.9892
 120000/ 200000: 2.1600
 130000/ 200000: 2.0309
 140000/ 200000: 2.2675
 150000/ 200000: 2.0170
 160000/ 200000: 2.5236
 170000/ 200000: 2.1743
 180000/ 200000: 1.8421
 190000/ 200000: 2.2615
